# Documents and downloads

Pick one document from a filing, look inside a source package, and save one
company's filings to a folder with a manifest.

## Identify your SEC requests

The SEC asks automated clients for a name or organization and a contact email,
separated by a space. For example, `Alex Morgan alex@example.com` or
`Example Research filings@example.com`. These are examples: use your own name
and a working email. No SEC account or API key is needed.

In the next cell, uncomment the `os.environ[...]` line and replace the example
with your details. If you already set this environment variable in your
terminal, leave the line commented. The label is sent to the SEC with each
request. See the [setup guide](../../docs/getting-started.md#sec-request-identity).

In [1]:
import os
from collections import Counter

from opencorpfilings import Company, provider_info, search_companies

# Uncomment this line and replace the example with your own name and contact email.
# os.environ["OPENCORPFILINGS_SEC_USER_AGENT"] = "Alex Morgan alex@example.com"
assert os.getenv("OPENCORPFILINGS_SEC_USER_AGENT"), "Uncomment the SEC identity example above and replace it with your own details, or set OPENCORPFILINGS_SEC_USER_AGENT in your environment."

SEC exhibits keep their EDGAR type, for example Exhibit 21 (subsidiaries):

In [2]:
filing = Company("AAPL", market="US").filings(form="annual").latest()
exhibits = [document for document in filing.documents if document.role == "exhibit"]
[(document.document_type, document.filename) for document in exhibits]

[('EX-21.1', 'a10-kexhibit21109272025.htm'), ('EX-23.1', 'a10-kexhibit23109272025.htm'), ('EX-31.1', 'a10-kexhibit31109272025.htm'), ('EX-31.2', 'a10-kexhibit31209272025.htm'), ('EX-32.1', 'a10-kexhibit32109272025.htm'), ('EX-4.1', 'a10-kexhibit4109272025.htm')]

In [3]:
subsidiaries = filing.select_document(document_type="EX-21.1")
subsidiaries.text()[:300]

'EX-21.1\n3\na10-kexhibit21109272025.htm\nEX-21.1\nExhibit 21.1\nSubsidiaries of\nApple Inc.*\nJurisdiction\nof Incorporation\nApple Asia Limited\tHong Kong\nApple Asia LLC\tDelaware, U.S.\nApple Canada Inc.\tCanada\nApple Computer Trading (Shanghai) Co., Ltd.\tChina\nApple Distribution International Limited\tIreland\n'

A French annual report arrives as an ESEF package. `members` lists what is inside:

In [4]:
report = Company("969500XXRPGD7HCAFA90", market="FR").filings(form="annual").latest()
package = report.primary_document
[(member.entry_path, member.size) for member in package.members][:6]

[('Legrand-2025-12-31-1-FR/META-INF/taxonomyPackage.xml', 1082), ('Legrand-2025-12-31-1-FR/reports/Legrand-2025-12-31-1-FR.xhtml', 91246552), ('Legrand-2025-12-31-1-FR/www.legrandgroup.com/Legrand-2025-12-31_cal.xml', 51696), ('Legrand-2025-12-31-1-FR/www.legrandgroup.com/Legrand-2025-12-31_def.xml', 132931), ('Legrand-2025-12-31-1-FR/META-INF/catalog.xml', 362), ('Legrand-2025-12-31-1-FR/www.legrandgroup.com/Legrand-2025-12-31_lab-fr.xml', 21833)]

Save one company's filings to a folder. A bound is required (`limit=`, dates or
`all_history=True`); `manifest.json` records each file's source URL and SHA-256.

In [5]:
greggs = Company("00502851", market="UK")
result = greggs.filings(form="annual", limit=2).download("downloads/greggs")
result

{'manifest': 'downloads\\greggs\\manifest.json', 'filings': 2, 'downloaded': 2, 'skipped': 0, 'failed': 0}

Running it again skips files that still match the manifest:

In [6]:
greggs.filings(form="annual", limit=2).download("downloads/greggs")

{'manifest': 'downloads\\greggs\\manifest.json', 'filings': 2, 'downloaded': 0, 'skipped': 2, 'failed': 0}

`file_types` keeps documents with chosen extensions, across all of a filing's documents:

In [7]:
apple = Company("AAPL", market="US")
apple.filings(form="annual", limit=1).download("downloads/apple-xbrl", file_types=(".xsd", ".xml"))

{'manifest': 'downloads\\apple-xbrl\\manifest.json', 'filings': 1, 'downloaded': 7, 'skipped': 0, 'failed': 0}